In [1]:
import pandas as pd
import numpy as np

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
from config import raw_data, control_data, external_data, raw_kyiv, filtered_data, daily_data

#### text

In [3]:
df_text = pd.read_csv(filtered_data / "text_messages_filtered.csv")

df_text["datetime"] = pd.to_datetime(df_text["datetime"])
df_text["date"] = df_text["datetime"].dt.date
df_text["hour"] = df_text["datetime"].dt.hour
df_text["weekday"] = df_text["datetime"].dt.weekday 

df_text.head(2)

,donation_id,id,conversation_id,sender_id,datetime,word_count,donor_sender_id,is_donor,date,hour,weekday
0,06b923ca-4d2d-44bb-8b20-4e2c9b08ce9f,e39038a3-9136-4e60-bd59-7405cdaa8953,01842182-4154-4143-b10d-32c21307f4fd,50475d11-75e7-4d96-a0e5-5c62d7f67c3a,2022-05-22 22:47:59.730,1,b5518648-ab76-42f8-8a2b-f539f9d9cf47,0,2022-05-22,22,6
1,06b923ca-4d2d-44bb-8b20-4e2c9b08ce9f,7157c4a6-8137-4c4e-849d-03628c89a001,01842182-4154-4143-b10d-32c21307f4fd,50475d11-75e7-4d96-a0e5-5c62d7f67c3a,2022-05-22 22:48:10.917,5,b5518648-ab76-42f8-8a2b-f539f9d9cf47,0,2022-05-22,22,6


In [4]:
donor_messages = df_text[df_text["is_donor"] == 1]

word count


In [5]:
daily_wordcount = (donor_messages
        .groupby(["donation_id", "date"])["word_count"]
        .sum()
        .reset_index()
        .rename(columns={"word_count": "donor_daily_word_count"})
)

message count

In [6]:
daily_msgcount = (donor_messages
        .groupby(["donation_id", "date"])
        .size()
        .reset_index(name="donor_daily_message_count")
)

active chats

In [7]:
daily_active_chats = (df_text
        .groupby(["donation_id", "date"])["conversation_id"]
        .nunique()
        .reset_index()
        .rename(columns={"conversation_id": "daily_active_chats"})
)

active hours

In [8]:
daily_active_hours = (donor_messages
        .groupby(["donation_id", "date"])["hour"]
        .nunique()
        .reset_index()
        .rename(columns={"hour": "donor_daily_active_hours"})
)

average length

In [9]:
daily_avg_length = (
    donor_messages
    .groupby(["donation_id", "date"])["word_count"]
    .mean()
    .reset_index()
    .rename(columns={"word_count": "donor_daily_avg_length"})
)

time shares of text messages 

In [10]:
donor_messages["is_night"] = ((donor_messages["hour"] >= 0) & 
                              (donor_messages["hour"] < 6)).astype(int)

donor_messages["is_morning"] = ((donor_messages["hour"] >= 6) & 
                                (donor_messages["hour"] < 12)).astype(int)

donor_messages["is_afternoon"] = ((donor_messages["hour"] >= 12) & 
                                  (donor_messages["hour"] < 18)).astype(int)

donor_messages["is_evening"] = ((donor_messages["hour"] >= 18) & 
                                (donor_messages["hour"] < 24)).astype(int)


daily_time_shares = (
    donor_messages
    .groupby(["donation_id", "date"])[
        ["is_morning", "is_afternoon", "is_evening", "is_night"]
    ]
    .mean()
    .reset_index()
    .rename(columns={
        "is_morning": "morning_share",
        "is_afternoon": "afternoon_share",
        "is_evening": "evening_share",
        "is_night": "night_share"
    })
)

/var/folders/vx/kbkdd94d7092nh7qby2pf8lr0000gn/T/ipykernel_78640/598385715.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  donor_messages["is_night"] = ((donor_messages["hour"] >= 0) &
/var/folders/vx/kbkdd94d7092nh7qby2pf8lr0000gn/T/ipykernel_78640/598385715.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  donor_messages["is_morning"] = ((donor_messages["hour"] >= 6) &
/var/folders/vx/kbkdd94d7092nh7qby2pf8lr0000gn/T/ipykernel_78640/598385715.py:7: SettingWithCopyWarning: 
A value is trying to be se

In [11]:
received_messages = df_text[df_text['is_donor'] == 0]

In [12]:
chat_to_donor = (
    df_text[df_text['is_donor'] == 1][['donation_id', 'conversation_id']]
    .drop_duplicates()
    .rename(columns={'donation_id': 'donor_id'}) 
)
received_with_donor = received_messages.merge(
    chat_to_donor,
    on='conversation_id',
    how='inner'
)

received messages

In [13]:
n_messages_received = (
    received_with_donor
    .groupby(['donor_id', 'date'])
    .size()
    .reset_index(name='n_messages_received')
    .rename(columns={'donor_id': 'donation_id'}) 
)

received words

In [14]:
words_received = (
    received_with_donor
    .groupby(['donor_id', 'date'])['word_count']
    .sum()
    .reset_index()
    .rename(columns={'donor_id': 'donation_id', 'word_count': 'words_received'})
)

frac of sent words to top5 chats

In [15]:
top5_chats = (
    donor_messages
    .groupby(['donation_id', 'conversation_id'])['word_count']
    .sum()
    .reset_index()
    .sort_values(['donation_id', 'word_count'], ascending=[True, False])
    .groupby('donation_id')
    .head(5)
)[['donation_id', 'conversation_id']]

words_in_top5 = (
    donor_messages
    .merge(top5_chats, on=['donation_id', 'conversation_id'], how='inner')
    .groupby(['donation_id', 'date'])['word_count']
    .sum()
    .reset_index()
    .rename(columns={'word_count': 'words_in_top5'})
)

words_total = (
    donor_messages
    .groupby(['donation_id', 'date'])['word_count']
    .sum()
    .reset_index()
    .rename(columns={'word_count': 'words_total'})
)

frac_words_top5 = words_total.merge(
    words_in_top5, on=['donation_id', 'date'], how='left'
).fillna(0)

frac_words_top5['frac_words_closest_5_contacts'] = (
    frac_words_top5['words_in_top5'] /
    (frac_words_top5['words_total'] + 1e-9)
)

frac_words_closest_5 = frac_words_top5[[
    'donation_id', 'date', 'frac_words_closest_5_contacts'
]]


#### comments

In [16]:
comments_path = filtered_data / "comments_sorted.csv"

df_comments = pd.read_csv(comments_path)

df_comments["datetime"] = pd.to_datetime(df_comments["datetime"])
df_comments["date"] = df_comments["datetime"].dt.date

comment count

In [17]:
daily_comment_count = (
    df_comments
        .groupby(["donation_id", "date"])
        .size()
        .reset_index(name="donor_daily_comment_count")
)

average comment length

In [18]:
daily_comment_avg_length = (
    df_comments
        .groupby(["donation_id", "date"])["word_count"]
        .mean()
        .reset_index()
        .rename(columns={"word_count": "donor_daily_comment_avg_length"})
)

In [19]:
daily_comments = daily_comment_count.merge(
    daily_comment_avg_length,
    on=["donation_id", "date"],
    how="left"
)

#### reactions

In [20]:
reactions_path = filtered_data / "reactions_sorted.csv"

df_reactions = pd.read_csv(reactions_path)
df_reactions["datetime"] = pd.to_datetime(df_reactions["datetime"])
df_reactions["date"] = df_reactions["datetime"].dt.date

reaction count

In [21]:
daily_reactions = (
    df_reactions
        .groupby(["donation_id", "date"])
        .size()
        .reset_index(name="donor_daily_reaction_count")
)

night share reactions

In [22]:
df_reactions['hour'] = pd.to_datetime(df_reactions['datetime']).dt.hour
df_reactions['is_night'] = ((df_reactions['hour'] >= 0) &
                            (df_reactions['hour'] < 6)).astype(int)

night_share_reactions = (
    df_reactions
    .groupby(['donation_id', 'date'])['is_night']
    .mean()
    .reset_index()
    .rename(columns={'is_night': 'night_share_reactions'})
)

#### posts 

In [23]:
posts_path = filtered_data / "posts_sorted.csv"

df_posts = pd.read_csv(posts_path)

df_posts["datetime"] = pd.to_datetime(df_posts["datetime"])
df_posts["date"] = df_posts["datetime"].dt.date

posts count

In [24]:
daily_post_count = (
    df_posts
        .groupby(["donation_id", "date"])
        .size()
        .reset_index(name="donor_daily_post_count")
)

average length of the text in posts

In [25]:
daily_post_avg_length = (
    df_posts
        .groupby(["donation_id", "date"])["word_count"]
        .mean()
        .reset_index()
        .rename(columns={"word_count": "donor_daily_post_avg_length"})
)

total media 

In [26]:
daily_post_total_media = (
    df_posts
        .groupby(["donation_id", "date"])["media_count"]
        .sum()
        .reset_index()
        .rename(columns={"media_count": "donor_daily_post_total_media"})
)

In [27]:
daily_posts = (
    daily_post_count
        .merge(daily_post_avg_length, on=["donation_id", "date"], how="left")
        .merge(daily_post_total_media, on=["donation_id", "date"], how="left")
)

#### audio

In [28]:
df_audio = pd.read_csv(filtered_data / "audio_messages_filtered.csv")

df_audio["datetime"] = pd.to_datetime(df_audio["datetime"])
df_audio["date"] = df_audio["datetime"].dt.date

df_audio_donor = df_audio[df_audio["is_donor"] == 1]


In [29]:
df_audio

,donation_id,id,conversation_id,sender_id,datetime,length_seconds,donor_sender_id,is_donor,date
0,06b923ca-4d2d-44bb-8b20-4e2c9b08ce9f,5adde0cc-8de5-4b8a-bf60-4825d55dd653,03313ca6-c08c-402c-89da-1c6eefa7ac20,b5518648-ab76-42f8-8a2b-f539f9d9cf47,2023-10-31 20:09:41.402,4,b5518648-ab76-42f8-8a2b-f539f9d9cf47,1,2023-10-31
1,06b923ca-4d2d-44bb-8b20-4e2c9b08ce9f,f4d0c7c3-8d98-4198-a99f-e3395e8c2a83,03313ca6-c08c-402c-89da-1c6eefa7ac20,50475d11-75e7-4d96-a0e5-5c62d7f67c3a,2023-10-31 20:46:30.854,30,b5518648-ab76-42f8-8a2b-f539f9d9cf47,0,2023-10-31
2,06b923ca-4d2d-44bb-8b20-4e2c9b08ce9f,b8efd3a1-8cac-46c3-853e-912a5fa6ac54,03313ca6-c08c-402c-89da-1c6eefa7ac20,50475d11-75e7-4d96-a0e5-5c62d7f67c3a,2023-12-18 22:39:20.515,22,b5518648-ab76-42f8-8a2b-f539f9d9cf47,0,2023-12-18
3,06b923ca-4d2d-44bb-8b20-4e2c9b08ce9f,82b8b974-82a0-4e17-a343-5b6b513968ba,03313ca6-c08c-402c-89da-1c6eefa7ac20,50475d11-75e7-4d96-a0e5-5c62d7f67c3a,2023-12-18 22:39:32.331,9,b5518648-ab76-42f8-8a2b-f539f9d9cf47,0,2023-12-18
4,06b923ca-4d2d-44bb-8b20-4e2c9b08ce9f,ab6d5424-b66d-4e97-89c0-9671c19cc4d7,033f62e2-a44d-4ebd-b41f-1a88eed1fbc2,b50d541e-f39a-4578-8f9e-f03b40fd6d02,2022-09-05 22:46:00.341,4,b5518648-ab76-42f8-8a2b-f539f9d9cf47,0,2022-09-05
...,...,...,...,...,...,...,...,...,...
100253,f96c5581-8bdb-4792-a7a6-3dc5bc5c8f16,93852766-6a9c-40ec-a24b-a60cb6359e8f,fa054103-a357-4f03-9302-3e431d4462c7,f77511c4-7b39-49f8-b0a7-662e926e4e57,2023-12-04 14:27:09.529,4,4fd30dcd-5bd1-464e-9eec-6ed59e2bc61e,0,2023-12-04
100254,f96c5581-8bdb-4792-a7a6-3dc5bc5c8f16,c6abc0d5-f0e6-4121-8eca-7b7866cc6f55,fa054103-a357-4f03-9302-3e431d4462c7,f77511c4-7b39-49f8-b0a7-662e926e4e57,2023-12-04 14:27:30.546,19,4fd30dcd-5bd1-464e-9eec-6ed59e2bc61e,0,2023-12-04
100255,f96c5581-8bdb-4792-a7a6-3dc5bc5c8f16,3668f7e5-1e00-415f-8fac-1d951881d6e4,fa7489a1-0f10-4d58-a213-908e3003a954,336f951a-b031-461a-a1c4-bb2c006de9df,2021-09-09 16:02:46.416,15,4fd30dcd-5bd1-464e-9eec-6ed59e2bc61e,0,2021-09-09
100256,f96c5581-8bdb-4792-a7a6-3dc5bc5c8f16,5e11d8a4-b976-4ec5-97af-86a7aaa71100,faac9bb2-f906-44e4-854e-0eb624876ef8,4fd30dcd-5bd1-464e-9eec-6ed59e2bc61e,2022-02-17 18:07:50.524,15,4fd30dcd-5bd1-464e-9eec-6ed59e2bc61e,1,2022-02-17


audio count

In [30]:
daily_audio_count = (
    df_audio_donor
    .groupby(["donation_id", "date"])
    .size()
    .reset_index(name="donor_daily_audio_count")
)

average length audio

In [31]:
audio_avg_length = (
    df_audio_donor
    .groupby(["donation_id", "date"])
    .agg(
        donor_daily_audio_count=("length_seconds", "count"),
        donor_daily_audio_avg_length=("length_seconds", "mean"),
    )
    .reset_index()
)

In [32]:
save_base = daily_data

files_to_save = {
    'donor_daily_word_count.csv': daily_wordcount,
    'donor_daily_message_count.csv': daily_msgcount,
    'daily_active_chats.csv': daily_active_chats,
    'donor_daily_active_hours.csv': daily_active_hours,
    'donor_daily_avg_length.csv': daily_avg_length,
    'donor_daily_time_shares.csv': daily_time_shares,
    'donor_daily_messages_received.csv': n_messages_received,
    'donor_daily_words_received.csv': words_received,
    'donor_daily_frac_words_top5.csv': frac_words_closest_5,
    'donor_daily_comments.csv': daily_comments,
    'donor_daily_reactions.csv': daily_reactions,
    'donor_daily_night_share_reactions.csv': night_share_reactions,
    'donor_daily_posts.csv': daily_posts,
    'donor_daily_audio.csv': daily_audio_count,
    'donor_daily_audio_avg_length.csv': audio_avg_length
    
}

for filename, df in files_to_save.items():
    df.to_csv(str(save_base / filename), index=False)